In [1]:
from tensorflow.keras import layers, models
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

In [2]:
iris = load_iris()
X, y = iris.data, iris.target.reshape(-1,1)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)
encoder = OneHotEncoder(sparse_output=False)
y_cat = encoder.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X_norm, y_cat, test_size=0.2, random_state=42)

In [3]:
tf_model = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax')
])
tf_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
tf_model.fit(X_train, y_train, epochs=50, verbose=0)
loss, acc = tf_model.evaluate(X_test, y_test, verbose=0)
print(f"TensorFlow Loss Amount: {loss:.4f}")
print(f"TensorFlow Test Accuracy: {acc:.4f}")

TensorFlow Loss Amount: 0.6824
TensorFlow Test Accuracy: 0.8000


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
le = LabelEncoder()
y_idx = le.fit_transform(iris.target)
X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    torch.tensor(X_norm, dtype=torch.float32),
    torch.tensor(y_idx, dtype=torch.long),
    test_size=0.2,
    random_state=42
)

In [6]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 3)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [7]:
pt_model = IrisNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.01)

In [8]:
for epoch in range(100):
    optimizer.zero_grad()
    outputs = pt_model(X_train_pt)
    loss = criterion(outputs, y_train_pt)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1} : Loss = {loss:.4f}")

Epoch 1 : Loss = 1.1407
Epoch 2 : Loss = 1.1167
Epoch 3 : Loss = 1.0935
Epoch 4 : Loss = 1.0711
Epoch 5 : Loss = 1.0494
Epoch 6 : Loss = 1.0285
Epoch 7 : Loss = 1.0082
Epoch 8 : Loss = 0.9880
Epoch 9 : Loss = 0.9678
Epoch 10 : Loss = 0.9473
Epoch 11 : Loss = 0.9263
Epoch 12 : Loss = 0.9045
Epoch 13 : Loss = 0.8818
Epoch 14 : Loss = 0.8583
Epoch 15 : Loss = 0.8339
Epoch 16 : Loss = 0.8087
Epoch 17 : Loss = 0.7830
Epoch 18 : Loss = 0.7569
Epoch 19 : Loss = 0.7305
Epoch 20 : Loss = 0.7041
Epoch 21 : Loss = 0.6779
Epoch 22 : Loss = 0.6521
Epoch 23 : Loss = 0.6270
Epoch 24 : Loss = 0.6029
Epoch 25 : Loss = 0.5798
Epoch 26 : Loss = 0.5578
Epoch 27 : Loss = 0.5371
Epoch 28 : Loss = 0.5178
Epoch 29 : Loss = 0.4998
Epoch 30 : Loss = 0.4831
Epoch 31 : Loss = 0.4676
Epoch 32 : Loss = 0.4534
Epoch 33 : Loss = 0.4402
Epoch 34 : Loss = 0.4280
Epoch 35 : Loss = 0.4166
Epoch 36 : Loss = 0.4061
Epoch 37 : Loss = 0.3963
Epoch 38 : Loss = 0.3872
Epoch 39 : Loss = 0.3786
Epoch 40 : Loss = 0.3705
Epoch 41 

In [9]:
with torch.no_grad():
    preds = pt_model(X_test_pt).argmax(dim=1)
    acc_pt = (preds == y_test_pt).float().mean().item()
print(f"PyTorch Test Accuracy: {acc_pt:.4f}")

PyTorch Test Accuracy: 0.9667
